# CS781 Phase 3 - Rhetorical Role (RR) - Model 4 (GQA + MHA mix, MoE) + LoRA

Same pipeline, folds, class-imbalance handling and training recipe as the Model 1 notebooks, with the **Model-4 backbone** (GQA + full-MHA layers, MoE feed-forward) and **LoRA** on all 7 linear layers (attention `q/k/v/o` and `gate/up/down` of every MoE expert and the shared expert; the router stays frozen).

**Attach as inputs**
1. Output of the **RR data-prep notebook** (`tokenized_rr.pkl`, `rr_utils.py`).
2. Output of your **Model-4 Phase-1 notebook**: the folder `legal-gqamoe-final` (`config.json` + `model.safetensors`) **and** the three code files that notebook wrote next to it: `model.py`, `moe.py`, `legal_gqa_model.py`. If they are not in that output, put the three files in a small dataset and attach it (they must be in one folder).

**Same as the Model 1 runs:** document chunks with a SEP after every sentence, sentence vector = SEP state + mean of token states, position + legal-cue features, domain embedding, BiGRU over the sentences of a chunk, joint CL+IT training, 5-fold CV with the same folds (so the out-of-fold predictions of all models line up for the later ensemble), inverse-sqrt class weights, rare-chunk oversampling, label smoothing, multi-cut test averaging.

**Post-processing (upgraded, scored held-out):** `tau` / `alpha` as before, plus an optional per-label bias and a document-level BiGRU second stage. Every candidate is tuned on 4 folds and scored on the 5th; the table is printed and the best candidate writes the submission. The model-training settings are the same as in the first Model 1 runs (no EMA; distillation is off here: set `USE_KD = True` and attach a run with identical folds if you want it).

Outputs: `rr_model4_gqamoe_lora/` with `prediction.json`, `prediction_CL.json`, `prediction_IT.json`, `submission.zip`, `oof_report.json`, per-fold weights.

## 1. Settings

In [ ]:
import os, sys, glob, json, time, math, random, shutil, pickle, re, hashlib, zipfile, types
import numpy as np

T0 = time.time()

# ======================= SETTINGS =======================
EXPERIMENT_NAME = "rr_model4_gqamoe_lora"
SMOKE_TEST = False          # True = tiny fast run to check that everything works

PROTOCOL = "kfold"          # "kfold" (recommended) or "dev" (train on train, validate on dev)
N_FOLDS = 5
SEED = 42

# --- chunks ---
TRAIN_CHUNK_MIN = 512       # each epoch every document is re-cut with a random chunk length in [MIN, MAX]
TRAIN_CHUNK_MAX = 1024      # (different boundaries every epoch = augmentation). Must be <= Phase-1 context (1024)
INFER_CHUNK_LENS = [1024, 800, 640]   # test/OOF: average the log-probs of these different cuttings

# --- training ---
EPOCHS = 8
PATIENCE = 3                # stop a fold if validation macro-F1 does not improve for this many epochs
BATCH_CHUNKS = 4            # chunks per optimizer step
MICRO_BATCH = 2             # chunks per forward pass (lower it if you run out of GPU memory)
EVAL_BATCH = 4
LR_LORA = 2e-4
LR_HEAD = 1e-3
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.06
GRAD_CLIP = 1.0
LABEL_SMOOTHING = 0.02
USE_CHECKPOINT = True       # gradient checkpointing (saves GPU memory)
EMA_DECAY = 0.0            # moving average of the trainable weights (0 = off); the EMA weights are evaluated and saved

# --- class imbalance ---
CLASS_WEIGHTS = "sqrt"      # "none" | "sqrt" | "inv"   (computed from the training documents of each fold)
WEIGHT_CAP = 10.0           # largest allowed class weight
OVERSAMPLE_RARE = True      # repeat chunks that contain a rare label
RARE_MAX_COUNT = 320        # a label with at most this many training sentences is "rare"
OVERSAMPLE_EXTRA = 1        # how many extra copies of such chunks

# --- LoRA ---
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# --- head ---
HEAD = "bigru"              # "bigru" or "linear" (linear = no cross-sentence head, for the ablation)
USE_FEATURES = True         # position + legal cue features
HEAD_DIM = 512
GRU_HIDDEN = 256
HEAD_DROPOUT = 0.2
DOMAIN_EMB = 16

# --- post-processing (tuned on out-of-fold predictions) ---
TAU_GRID = [-0.5, -0.25, 0.0, 0.25, 0.5, 0.75, 1.0]   # prior correction strength
ALPHA_GRID = [0.0, 0.25, 0.5, 1.0]                    # transition weight in Viterbi

# --- distillation (stage B) ---
USE_TEACHER_RUN = False     # use an earlier run with identical folds as stage A instead of training stage A again
USE_KD = False              # stage B: students trained on the teacher's soft labels
KD_LAMBDA = 0.5             # weight of the soft-label loss (the rest is the normal class-weighted loss)
KD_TEMP = 2.0
KD_USE_TEST = True          # the unlabelled test documents also get soft labels (only the soft-label loss)

# --- extra post-processing ---
USE_CLASS_BIAS = True       # per-label and per-domain bias on top of tau / alpha
BIAS_STEPS = [-1.0, -0.5, 0.5, 1.0]
BIAS_ROUNDS = 2
BIAS_MIN_GAIN = 0.002       # a bias change must raise the macro-F1 by at least this much
BIAS_CLIP = 3.0
USE_STAGE2 = True           # document-level BiGRU on the log-probs (scored with nested cross-validation)
S2_HIDDEN = 48
S2_EPOCHS = 25
S2_LR = 2e-3
S2_SEEDS = 3
S2_DROPOUT = 0.3
S2_NOISE = 0.3
S2_BATCH = 8

# --- session ---
MAX_SESSION_HOURS = 10.0    # stop cleanly after this long (Kaggle commit limit is 12 h); progress is saved

# --- paths (None = search /kaggle/input automatically) ---
DATA_PKL = None
MODEL_DIR = None
MODEL_SRC = None            # path of model.py (moe.py and legal_gqa_model.py must be in the same folder)
RESUME_DIR = None           # output folder of an earlier run of THIS notebook
TEACHER_DIR = None          # folder of an earlier Model-1 LoRA run to use as stage A (None = search /kaggle/input)
OUT_DIR = "/kaggle/working/" + EXPERIMENT_NAME
# [[END OF SETTINGS]]

for _name in ["DATA_PKL", "MODEL_DIR", "MODEL_SRC", "RESUME_DIR", "OUT_DIR", "TEACHER_DIR"]:
    _value = os.environ.get("RR_" + _name)
    if _value:
        globals()[_name] = _value
if os.environ.get("RR_SMOKE") == "1":
    SMOKE_TEST = True
# (more env overrides, only used for local testing)
for _name in ["PROTOCOL", "HEAD", "CLASS_WEIGHTS"]:
    _value = os.environ.get("RR_" + _name)
    if _value:
        globals()[_name] = _value
if os.environ.get("RR_MAX_HOURS"):
    MAX_SESSION_HOURS = float(os.environ["RR_MAX_HOURS"])
if os.environ.get("RR_FEATURES") == "0":
    USE_FEATURES = False
if os.environ.get("RR_EPOCHS"):
    EPOCHS = int(os.environ["RR_EPOCHS"])
for _name in ["USE_KD", "USE_STAGE2", "USE_CLASS_BIAS", "USE_TEACHER_RUN"]:
    if os.environ.get("RR_" + _name) == "0":
        globals()[_name] = False
if os.environ.get("RR_EMA"):
    EMA_DECAY = float(os.environ["RR_EMA"])

MAX_DOCS = None
if SMOKE_TEST:
    EPOCHS = 2
    PATIENCE = 2
    N_FOLDS = 2
    BATCH_CHUNKS = 2
    MICRO_BATCH = 1
    EVAL_BATCH = 2
    TRAIN_CHUNK_MIN = 300
    TRAIN_CHUNK_MAX = 400
    INFER_CHUNK_LENS = [400, 300]
    HEAD_DIM = 64
    GRU_HIDDEN = 32
    LORA_R = 4
    S2_EPOCHS = 4
    S2_SEEDS = 1
    S2_HIDDEN = 16
    BIAS_ROUNDS = 1
    MAX_DOCS = 8
    TAU_GRID = [0.0, 0.5]
    ALPHA_GRID = [0.0, 0.5]

if os.environ.get("RR_EPOCHS"):
    EPOCHS = int(os.environ["RR_EPOCHS"])
    PATIENCE = max(PATIENCE, EPOCHS)

assert PROTOCOL in ("kfold", "dev")
assert HEAD in ("bigru", "linear")
assert CLASS_WEIGHTS in ("none", "sqrt", "inv")
os.makedirs(OUT_DIR, exist_ok=True)
print("SMOKE_TEST:", SMOKE_TEST, "| PROTOCOL:", PROTOCOL, "| OUT_DIR:", OUT_DIR)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import peft
except ImportError:
    os.system("pip install -q peft")
    import peft
from peft import LoraConfig, get_peft_model
print("peft", peft.__version__, "| torch", torch.__version__)

# Kaggle images can ship a torchao that is too old for the installed peft, which makes get_peft_model crash.
# We never use torchao, so tell peft it is not available.
try:
    peft.import_utils.is_torchao_available()
except ImportError as err:
    print("peft/torchao version clash ignored:", str(err)[:100])
    def _torchao_off():
        return False
    for _name, _mod in list(sys.modules.items()):
        if _name.startswith("peft") and hasattr(_mod, "is_torchao_available"):
            _mod.is_torchao_available = _torchao_off

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("device:", DEVICE)
if DEVICE.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def time_is_up():
    return (time.time() - T0) > MAX_SESSION_HOURS * 3600

## 2. Find data, model code and Phase-1 weights

In [ ]:
def find_files(filename):
    return sorted(glob.glob("/kaggle/input/**/" + filename, recursive=True))


# ---------- data ----------
if DATA_PKL is None:
    candidates = find_files("tokenized_rr.pkl")
    if len(candidates) == 0:
        raise FileNotFoundError("tokenized_rr.pkl not found. Attach the output of the RR data-prep notebook.")
    if len(candidates) == 1:
        DATA_PKL = candidates[0]
    else:
        # several data-prep outputs attached: take the one with the most test sentences, then the newest
        print("More than one tokenized_rr.pkl found:")
        ranked = []
        for p in candidates:
            fh = open(p, "rb")
            b = pickle.load(fh)
            fh.close()
            n_test = 0
            for dom in b["meta"]["domains"]:
                for doc in b["data"][dom]["test"]["docs"]:
                    n_test += len(doc["sentences"])
            print("   ", p, "| test sentences:", n_test)
            ranked.append((n_test, os.path.getmtime(p), p))
            del b
        ranked.sort()
        DATA_PKL = ranked[-1][2]
print("data:", DATA_PKL)

DATA_DIR = os.path.dirname(DATA_PKL)
utils_candidates = [os.path.join(DATA_DIR, "rr_utils.py")] + find_files("rr_utils.py")
UTILS_PATH = None
for p in utils_candidates:
    if os.path.exists(p):
        UTILS_PATH = p
        break
if UTILS_PATH is None:
    raise FileNotFoundError("rr_utils.py not found next to tokenized_rr.pkl.")
sys.path.insert(0, os.path.dirname(UTILS_PATH))
import rr_utils
print("rr_utils:", UTILS_PATH)


# ---------- Phase-1 weights ----------
def find_model_dir():
    if MODEL_DIR is not None:
        return MODEL_DIR
    found = []
    for cfg_path in find_files("config.json"):
        folder = os.path.dirname(cfg_path)
        try:
            model_type = json.load(open(cfg_path)).get("model_type")
        except Exception:
            continue
        has_weights = os.path.exists(os.path.join(folder, "model.safetensors")) or \
                      os.path.exists(os.path.join(folder, "pytorch_model.bin"))
        if model_type == "legal_gqa_mha_moe" and has_weights:
            found.append(folder)
    if len(found) == 0:
        raise FileNotFoundError("No Phase-1 Model-4 folder (model_type legal_gqa_mha_moe). Attach the Model-4 notebook output.")
    final_ones = []
    for d in found:
        if "checkpoint-" not in d:
            final_ones.append(d)
    if len(final_ones) > 0:
        return final_ones[0]
    return max(found, key=lambda d: int(d.rsplit("-", 1)[1]))


MODEL_PATH = find_model_dir()
print("Phase-1 model:", MODEL_PATH)

# ---------- model code ----------
def find_model_src():
    if MODEL_SRC is not None:
        return MODEL_SRC
    places = ["/kaggle/working/model.py", os.path.join(MODEL_PATH, "model.py"),
              os.path.join(os.path.dirname(MODEL_PATH), "model.py")]
    for moe_path in find_files("moe.py"):
        places.append(os.path.join(os.path.dirname(moe_path), "model.py"))
    for p in places:
        folder = os.path.dirname(p)
        if os.path.exists(p) and os.path.exists(os.path.join(folder, "moe.py")):
            return p
    raise FileNotFoundError(
        "model.py + moe.py (Model-4 code) not found in the same folder. Attach the output of your Model-4 Phase-1 notebook "
        "(it wrote them), or put model.py, moe.py and legal_gqa_model.py in one small dataset.")


SRC_PATH = find_model_src()
print("model code:", SRC_PATH)
sys.path.insert(0, os.path.dirname(SRC_PATH))
if not os.path.exists(os.path.join(os.path.dirname(SRC_PATH), "legal_gqa_model.py")):
    others = find_files("legal_gqa_model.py")
    if len(others) == 0:
        raise FileNotFoundError("legal_gqa_model.py (imported by model.py) not found.")
    sys.path.insert(0, os.path.dirname(others[0]))
    print("legal_gqa_model.py from:", others[0])
from model import LegalGQAMoEConfig, LegalGQAMoEForCausalLM

# ---------- resume from an earlier run of this notebook ----------
def copy_missing(src, dst):
    for root, dirs, files in os.walk(src):
        rel = os.path.relpath(root, src)
        target = os.path.join(dst, rel)
        os.makedirs(target, exist_ok=True)
        for name in files:
            out_path = os.path.join(target, name)
            if not os.path.exists(out_path):
                shutil.copy(os.path.join(root, name), out_path)


resume_src = RESUME_DIR
if resume_src is None:
    prev = find_files(EXPERIMENT_NAME + "/cv_state.json")
    if len(prev) > 0:
        resume_src = os.path.dirname(prev[-1])
if resume_src is not None and os.path.abspath(resume_src) != os.path.abspath(OUT_DIR):
    copy_missing(resume_src, OUT_DIR)
    print("Resumed files copied from:", resume_src)
else:
    print("No earlier run found, starting fresh.")

## 3. Load documents and build features

In [ ]:
f = open(DATA_PKL, "rb")
blob = pickle.load(f)
f.close()
meta = blob["meta"]
data = blob["data"]

label2id = meta["label2id"]
K = int(meta["num_labels"])
id2label = {}
for name in label2id:
    id2label[label2id[name]] = name
LABEL_NAMES = []
for c in range(K):
    LABEL_NAMES.append(id2label[c])

DOMAINS = meta["domains"]
DOMAIN_IDX = {}
for i in range(len(DOMAINS)):
    DOMAIN_IDX[DOMAINS[i]] = i
SEP_ID = int(meta["sep_id"])
PAD_ID = int(meta["pad_id"])
HEADER_IDS = meta["header_ids"]
MAX_SENT_TOK = int(meta["max_sent_tok"])
DOC_CHUNK_CTX = int(meta["doc_chunk_ctx"])
print("labels:", K, "| domains:", DOMAINS, "| SEP:", SEP_ID, "| PAD:", PAD_ID)

# ---------- legal cue features ----------
CUES = [
    ("versus", r"\s(?:v|vs)\.?\s+[A-Z(]", 0),
    ("reporter", r"\b(?:AIR|SCC|SCR|ITR|ITD|CLJ|Taxman|SCALE|WLR|All ER|Comp\.? ?Cas)\b", 0),
    ("year_bracket", r"[\(\[]\s*(?:18|19|20)\d\d\s*[\)\]]", 0),
    ("section", r"\bsections?\b|\bsec\.|\bs\.\s?\d", re.I),
    ("article", r"\barticles?\s+\d", re.I),
    ("act_year", r"\bAct\b,?\s+(?:of\s+)?(?:18|19|20)\d\d", 0),
    ("rule_reg", r"\b(?:rules?|regulations?|clauses?|sub-?sections?|proviso|schedule)\b", re.I),
    ("held", r"\bheld\b", re.I),
    ("conclude", r"\b(?:we hold|we are of the (?:view|opinion)|in our (?:view|opinion)|we find|we conclude|accordingly|therefore|thus|hence)\b", re.I),
    ("disposal", r"\b(?:dismiss|allowed|set aside|quash|disposed|remand|affirm|upheld)\w*", re.I),
    ("argue", r"\b(?:contend|submit|argu|urge|learned (?:senior )?counsel|learned advocate)\w*", re.I),
    ("petitioner_side", r"\b(?:petitioner|appellant|plaintiff|applicant|assessee|complainant)s?\b", re.I),
    ("respondent_side", r"\b(?:respondent|defendant|revenue|department|opposite part(?:y|ies)|accused)s?\b", re.I),
    ("issue_q", r"\b(?:whether|question|issue|points? for|framed)\b", re.I),
    ("dissent", r"\b(?:dissent|minority view|respectfully|I disagree|unable to agree)\b", re.I),
    ("overrule", r"\b(?:overrul|reversed|no longer good law|wrongly decided|bad in law)\w*", re.I),
    ("distinguish", r"\b(?:distinguish|not applicable|of no assistance|not attracted|inapplicable)\w*", re.I),
    ("relied", r"\b(?:relied|reliance|followed|approved|reiterated|affirmed|referred)\b", re.I),
    ("lower_court", r"\b(?:high court|tribunal|trial court|lower court|learned (?:judge|single|magistrate)|impugned|appellate|commissioner)\b", re.I),
    ("facts_verbs", r"\b(?:filed|entered into|agreed|incorporated|appointed|issued|passed|notice|dated)\b", re.I),
    ("list_marker", r"^\(?(?:\d+|[ivxlcIVXLC]+|[a-zA-Z])[\.\)]\s", 0),
    ("has_quote", r"[\"“”]", 0),
    ("money", r"(?:Rs\.?|₹|INR|\$)\s?[\d,]+|\blakhs?\b|\bcrores?\b", re.I),
    ("date", r"\b\d{1,2}[./-]\d{1,2}[./-]\d{2,4}\b|\b(?:January|February|March|April|May|June|July|August|September|October|November|December)\b", 0),
]
CUE_REGEX = []
for _name, _pattern, _flags in CUES:
    CUE_REGEX.append(re.compile(_pattern, _flags))
N_CUE = len(CUES) + 5          # regex cues + 5 numeric cues
N_POS = 4
FEAT_DIM = N_CUE + N_POS


def sentence_cue_features(text, n_tokens):
    vec = np.zeros(N_CUE, dtype=np.float32)
    k = 0
    for rx in CUE_REGEX:
        if rx.search(text) is not None:
            vec[k] = 1.0
        k += 1
    n_chars = max(len(text), 1)
    digits = 0
    uppers = 0
    for ch in text:
        if ch.isdigit():
            digits += 1
        if ch.isupper():
            uppers += 1
    vec[k] = digits / n_chars
    vec[k + 1] = uppers / n_chars
    vec[k + 2] = math.log1p(n_tokens) / 6.0
    if text.endswith("?") or text.endswith(":"):
        vec[k + 3] = 1.0
    pet = len(re.findall(r"\b(?:petitioner|appellant|plaintiff|applicant|assessee)s?\b", text, re.I))
    res = len(re.findall(r"\b(?:respondent|defendant|revenue|department)s?\b", text, re.I))
    if pet > res:
        vec[k + 4] = 1.0
    elif res > pet:
        vec[k + 4] = -1.0
    return vec


def build_doc_features(sentences, sent_ids):
    n = len(sentences)
    feats = np.zeros((n, FEAT_DIM), dtype=np.float32)
    norm = math.log1p(2000)
    for i in range(n):
        feats[i, :N_CUE] = sentence_cue_features(sentences[i], len(sent_ids[i]))
        feats[i, N_CUE + 0] = i / max(n - 1, 1)
        feats[i, N_CUE + 1] = math.log1p(i) / norm
        feats[i, N_CUE + 2] = math.log1p(n - 1 - i) / norm
        feats[i, N_CUE + 3] = math.log1p(n) / norm
    return feats


# ---------- document table ----------
DOCS = []          # labelled documents first, then test documents
for domain in DOMAINS:
    for split in ["train", "dev"]:
        for doc in data[domain][split]["docs"]:
            DOCS.append({"domain": domain, "split": split, "id": doc["id"], "sentences": doc["sentences"],
                         "sent_ids": doc["sent_ids"], "labels": doc["labels"], "n": len(doc["sentences"])})
if MAX_DOCS is not None:
    # smoke test: keep only a few documents of each (domain, split)
    keep = []
    seen = {}
    for d in DOCS:
        key = d["domain"] + d["split"]
        seen[key] = seen.get(key, 0) + 1
        if d["split"] == "train" and seen[key] <= MAX_DOCS - 3:
            keep.append(d)
        elif d["split"] == "dev" and seen[key] <= 3:
            keep.append(d)
    DOCS = keep
N_LAB = len(DOCS)

for domain in DOMAINS:
    for doc in data[domain]["test"]["docs"]:
        DOCS.append({"domain": domain, "split": "test", "id": doc["id"], "sentences": doc["sentences"],
                     "sent_ids": doc["sent_ids"], "labels": None, "n": len(doc["sentences"])})
N_TEST = len(DOCS) - N_LAB

t1 = time.time()
for d in DOCS:
    d["feats"] = build_doc_features(d["sentences"], d["sent_ids"])
print("labelled docs:", N_LAB, "| test docs:", N_TEST, "| feature dim:", FEAT_DIM, "| %.1fs" % (time.time() - t1))
if N_TEST == 0:
    print("NOTE: no test documents in this data file (test set not released yet). Training and OOF still run.")

TEST_IDX = list(range(N_LAB, len(DOCS)))

## 4. Metrics, class weights, prior correction and transition-Viterbi

In [ ]:
def confusion_matrix(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    flat = np.bincount(y_true * K + y_pred, minlength=K * K)
    return flat.reshape(K, K)


def scores_from_conf(conf):
    tp = np.diag(conf).astype(np.float64)
    fp = conf.sum(axis=0) - tp
    fn = conf.sum(axis=1) - tp
    precision = np.zeros(K)
    recall = np.zeros(K)
    f1 = np.zeros(K)
    for c in range(K):
        if tp[c] + fp[c] > 0:
            precision[c] = tp[c] / (tp[c] + fp[c])
        if tp[c] + fn[c] > 0:
            recall[c] = tp[c] / (tp[c] + fn[c])
        if precision[c] + recall[c] > 0:
            f1[c] = 2 * precision[c] * recall[c] / (precision[c] + recall[c])
    support = conf.sum(axis=1)
    return precision, recall, f1, support


def macro_from_conf(conf):
    precision, recall, f1, support = scores_from_conf(conf)
    present = support > 0
    out = {}
    out["f1_present"] = float(f1[present].mean()) if present.any() else 0.0
    out["f1_all13"] = float(f1.mean())
    out["precision_present"] = float(precision[present].mean()) if present.any() else 0.0
    out["recall_present"] = float(recall[present].mean()) if present.any() else 0.0
    out["precision_all13"] = float(precision.mean())
    out["recall_all13"] = float(recall.mean())
    return out


def class_counts(doc_idx_list):
    counts = np.zeros(K, dtype=np.float64)
    for di in doc_idx_list:
        counts += np.bincount(DOCS[di]["labels"], minlength=K)
    return counts


def make_class_weights(counts):
    w = np.zeros(K, dtype=np.float64)
    total = counts.sum()
    k_seen = float((counts > 0).sum())
    if CLASS_WEIGHTS == "none":
        for c in range(K):
            if counts[c] > 0:
                w[c] = 1.0
        return w
    for c in range(K):
        if counts[c] > 0:
            base = total / (k_seen * counts[c])
            if CLASS_WEIGHTS == "sqrt":
                w[c] = math.sqrt(base)
            else:
                w[c] = base
    # rescale so the average weight over training sentences is 1, then cap
    scale = total / (w * counts).sum()
    w = w * scale
    for c in range(K):
        if w[c] > WEIGHT_CAP:
            w[c] = WEIGHT_CAP
    return w


def log_prior_from_counts(counts):
    p = (counts + 1.0) / (counts.sum() + K)
    return np.log(p)


def log_transitions(doc_idx_list, domain):
    trans = np.zeros((K, K), dtype=np.float64)
    for di in doc_idx_list:
        d = DOCS[di]
        if d["domain"] != domain:
            continue
        lab = d["labels"]
        for i in range(len(lab) - 1):
            trans[lab[i], lab[i + 1]] += 1
    trans = trans + 0.5
    trans = trans / trans.sum(axis=1, keepdims=True)
    return np.log(trans)


def decode(logp, trans_log, log_prior, tau, alpha):
    emission = logp - tau * log_prior[None, :]
    n = emission.shape[0]
    if alpha == 0.0 or n == 1:
        return emission.argmax(axis=1)
    score = emission[0].copy()
    back = np.zeros((n, K), dtype=np.int64)
    pair = alpha * trans_log
    for i in range(1, n):
        cand = score[:, None] + pair
        back[i] = cand.argmax(axis=0)
        score = cand.max(axis=0) + emission[i]
    path = np.zeros(n, dtype=np.int64)
    path[n - 1] = int(score.argmax())
    for i in range(n - 1, 0, -1):
        path[i - 1] = back[i, path[i]]
    return path


# self-test of the helpers
_c = confusion_matrix([0, 1, 1, 2], [0, 1, 2, 2])
assert macro_from_conf(_c)["f1_present"] > 0
_lp = np.log(np.full((5, K), 1.0 / K))
assert len(decode(_lp, np.log(np.full((K, K), 1.0 / K)), np.log(np.full(K, 1.0 / K)), 0.0, 0.5)) == 5
print("helpers OK")

## 5. Chunks and batching

In [ ]:
def make_train_chunks(doc_idx_list, rng, rare_labels, teacher=None):
    chunks = []
    for di in doc_idx_list:
        d = DOCS[di]
        max_len = rng.randint(TRAIN_CHUNK_MIN, TRAIN_CHUNK_MAX)
        parts = rr_utils.build_doc_chunks(d["sent_ids"], d["labels"], HEADER_IDS[d["domain"]], SEP_ID,
                                          max_len, DOC_CHUNK_CTX, MAX_SENT_TOK)
        for c in parts:
            c["doc"] = di
            if teacher is not None and di in teacher:
                c["teacher"] = teacher[di][c["sent_idx"]]
            chunks.append(c)

    extra = []
    if OVERSAMPLE_RARE and len(rare_labels) > 0:
        for c in chunks:
            if np.isin(c["labels"], rare_labels).any():
                for _ in range(OVERSAMPLE_EXTRA):
                    extra.append(c)
    chunks = chunks + extra
    rng.shuffle(chunks)
    return chunks


def collate(chunks):
    B = len(chunks)
    T = 0
    S = 0
    for c in chunks:
        T = max(T, len(c["input_ids"]))
        S = max(S, len(c["end_pos"]))

    ids = np.full((B, T), PAD_ID, dtype=np.int64)
    end = np.zeros((B, S), dtype=np.int64)
    start = np.zeros((B, S), dtype=np.int64)
    smask = np.zeros((B, S), dtype=bool)
    labels = np.full((B, S), -100, dtype=np.int64)
    feats = np.zeros((B, S, FEAT_DIM), dtype=np.float32)
    dom = np.zeros(B, dtype=np.int64)
    has_teacher = False
    for c in chunks:
        if "teacher" in c:
            has_teacher = True
    if has_teacher:
        teacher_arr = np.zeros((B, S, K), dtype=np.float32)
        tmask = np.zeros((B, S), dtype=bool)

    for b in range(B):
        c = chunks[b]
        d = DOCS[c["doc"]]
        header_len = 1 + len(HEADER_IDS[d["domain"]])
        ids[b, :len(c["input_ids"])] = c["input_ids"]
        n_s = len(c["end_pos"])
        end[b, :n_s] = c["end_pos"]
        for k in range(n_s):
            if k == 0:
                start[b, k] = header_len
            else:
                start[b, k] = c["end_pos"][k - 1] + 1
        smask[b, :n_s] = True
        labels[b, :n_s] = c["labels"]
        feats[b, :n_s, :] = d["feats"][c["sent_idx"]]
        dom[b] = DOMAIN_IDX[d["domain"]]
        if has_teacher and "teacher" in c:
            teacher_arr[b, :n_s, :] = c["teacher"]
            tmask[b, :n_s] = c["is_target"]

    batch = {}
    batch["input_ids"] = torch.from_numpy(ids)
    batch["end"] = torch.from_numpy(end)
    batch["start"] = torch.from_numpy(start)
    batch["smask"] = torch.from_numpy(smask)
    batch["labels"] = torch.from_numpy(labels)
    batch["feats"] = torch.from_numpy(feats)
    batch["dom"] = torch.from_numpy(dom)
    if has_teacher:
        batch["teacher"] = torch.from_numpy(teacher_arr)
        batch["tmask"] = torch.from_numpy(tmask)
    return batch


def to_device(batch):
    out = {}
    for key in batch:
        out[key] = batch[key].to(DEVICE)
    return out


# quick self-test on the first labelled document
_d0 = DOCS[0]
_parts = rr_utils.build_doc_chunks(_d0["sent_ids"], _d0["labels"], HEADER_IDS[_d0["domain"]], SEP_ID,
                                   INFER_CHUNK_LENS[-1], DOC_CHUNK_CTX, MAX_SENT_TOK)
for _c in _parts:
    _c["doc"] = 0
_b = collate(_parts[:2])
for _row in range(_b["input_ids"].shape[0]):
    for _k in range(int(_b["smask"][_row].sum())):
        assert int(_b["input_ids"][_row, int(_b["end"][_row, _k])]) == SEP_ID
        assert int(_b["start"][_row, _k]) <= int(_b["end"][_row, _k])
print("collate OK | chunks in first doc:", len(_parts), "| batch shape", tuple(_b["input_ids"].shape))

## 6. Model: Phase-1 Model-4 backbone (GQA/MHA + MoE) + LoRA + sentence head

In [ ]:
from torch.utils.checkpoint import checkpoint as torch_checkpoint


class RRModel(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        d = backbone.config.n_embd
        self.d = d

        self.sent_norm = nn.LayerNorm(2 * d)
        self.sent_drop = nn.Dropout(HEAD_DROPOUT)
        self.sent_proj = nn.Linear(2 * d, HEAD_DIM)

        self.feat_proj = nn.Linear(FEAT_DIM, 64)
        self.dom_emb = nn.Embedding(len(DOMAINS), DOMAIN_EMB)
        self.mix = nn.Linear(HEAD_DIM + 64 + DOMAIN_EMB, HEAD_DIM)
        self.mix_drop = nn.Dropout(HEAD_DROPOUT)

        if HEAD == "bigru":
            self.gru = nn.GRU(HEAD_DIM, GRU_HIDDEN, batch_first=True, bidirectional=True)
            out_dim = 2 * GRU_HIDDEN + HEAD_DIM
        else:
            out_dim = HEAD_DIM
        self.out_drop = nn.Dropout(HEAD_DROPOUT)
        self.cls = nn.Linear(out_dim, K)

    def hidden(self, input_ids):
        # same computation as the Phase-1 forward(), stopping before lm_head
        b = self.backbone
        T = input_ids.shape[1]
        x = b.drop(b.wte(input_ids))
        cos, sin = b._rope_slice(0, T, input_ids.device, x.dtype)
        for blk in b.blocks:
            if self.training and USE_CHECKPOINT:
                def run(h, blk=blk):
                    return blk(h, cos, sin, None, True, None, False)[0]
                x = torch_checkpoint(run, x, use_reentrant=False)
            else:
                x = blk(x, cos, sin, None, True, None, False)[0]     # causal; right padding needs no mask
        return b.ln_f(x)

    def forward(self, batch):
        ids = batch["input_ids"]
        end = batch["end"]
        start = batch["start"]
        smask = batch["smask"]

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            h = self.hidden(ids)
        h = h.float()
        B, T, d = h.shape
        S = end.shape[1]

        # SEP state of every sentence
        sep_state = torch.gather(h, 1, end.unsqueeze(-1).expand(B, S, d))

        # mean of the sentence's own tokens: tokens start .. end-1  (cumulative-sum trick)
        zeros = torch.zeros(B, 1, d, device=h.device, dtype=h.dtype)
        cs = torch.cat([zeros, torch.cumsum(h, dim=1)], dim=1)
        cs_end = torch.gather(cs, 1, end.unsqueeze(-1).expand(B, S, d))
        cs_start = torch.gather(cs, 1, start.unsqueeze(-1).expand(B, S, d))
        length = (end - start).clamp(min=1).unsqueeze(-1).float()
        mean_state = (cs_end - cs_start) / length

        x = torch.cat([sep_state, mean_state], dim=-1)
        x = self.sent_drop(self.sent_norm(x))
        x = F.gelu(self.sent_proj(x))

        if USE_FEATURES:
            f = F.gelu(self.feat_proj(batch["feats"]))
        else:
            f = torch.zeros(B, S, 64, device=h.device, dtype=h.dtype)
        dm = self.dom_emb(batch["dom"]).unsqueeze(1).expand(B, S, DOMAIN_EMB)
        x = F.gelu(self.mix(torch.cat([x, f, dm], dim=-1)))
        x = self.mix_drop(x)

        if HEAD == "bigru":
            lengths = smask.sum(dim=1).cpu()
            packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
            out, _ = self.gru(packed)
            out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=S)
            out = torch.cat([out, x], dim=-1)
        else:
            out = x
        return self.cls(self.out_drop(out)).float()


LOAD_REPORTED = False


def build_model(identity_check=False):
    global LOAD_REPORTED
    backbone, info = LegalGQAMoEForCausalLM.from_pretrained(MODEL_PATH, output_loading_info=True)
    if not LOAD_REPORTED:
        missing = []
        for key in info.get("missing_keys", []):
            if key != "lm_head.weight":          # tied to wte
                missing.append(key)
        unexpected = list(info.get("unexpected_keys", []))
        print("Phase-1 weights loaded | missing keys:", len(missing), "| unexpected keys:", len(unexpected))
        if len(missing) > 0:
            raise RuntimeError("Phase-1 weights were NOT fully loaded, missing: " + str(missing[:5]))
        if backbone.config.vocab_size <= max(SEP_ID, 0):
            raise RuntimeError("tokenizer / model vocab mismatch")
        LOAD_REPORTED = True
    backbone.config.use_cache = False
    backbone.float()                            # keep the experts and the adapters in one dtype

    if identity_check:
        probe = RRModel.__new__(RRModel)
        nn.Module.__init__(probe)
        probe.backbone = backbone
        ids = torch.randint(0, backbone.config.vocab_size, (1, 40))
        backbone.eval()
        with torch.no_grad():
            h_before = RRModel.hidden(probe, ids).clone()

    # some peft / transformers versions ask the model for its input embedding, give it a safe answer
    try:
        backbone.get_input_embeddings()
    except NotImplementedError:
        backbone.get_input_embeddings = types.MethodType(lambda self: self.wte, backbone)

    lora_cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                          target_modules=LORA_TARGETS, bias="none")
    get_peft_model(backbone, lora_cfg)          # wraps the linear layers of `backbone` in place and freezes the base
    n_wrapped = 0
    for name, _ in backbone.named_modules():
        if name.endswith("lora_A"):
            n_wrapped += 1
    if n_wrapped == 0:
        raise RuntimeError("LoRA wrapped no layers, check LORA_TARGETS")

    if identity_check:
        with torch.no_grad():
            h_after = RRModel.hidden(probe, ids)
        diff = float((h_before - h_after).abs().max())
        print("LoRA wrapped", n_wrapped, "linear layers | identity check max |diff| vs Phase-1:", "%.2e" % diff)
        if diff > 1e-3:
            raise RuntimeError("LoRA layers do not start at the identity")

    model = RRModel(backbone)
    model.to(DEVICE)
    return model


def trainable_names(model):
    names = []
    for name, p in model.named_parameters():
        if p.requires_grad:
            names.append(name)
    return names


def get_trainable_state(model):
    state = {}
    for name, p in model.named_parameters():
        if p.requires_grad:
            state[name] = p.detach().cpu().clone()
    return state


def load_trainable_state(model, state):
    for name, p in model.named_parameters():
        if name in state:
            p.data.copy_(state[name].to(p.device))


_m = build_model(identity_check=True)
total = 0
trainable = 0
for _n, _p in _m.named_parameters():
    total += _p.numel()
    if _p.requires_grad:
        trainable += _p.numel()
print("total %.2fM | trainable %.2fM (%.2f%%)" % (total / 1e6, trainable / 1e6, 100.0 * trainable / total))
del _m
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

## 7. Prediction and training (hard labels, EMA, optional distillation)

In [ ]:
def predict_docs(model, doc_list, chunk_lens):
    """Average log-probabilities of several different chunk cuttings. Returns {doc_index: (n, K) array}."""
    model.eval()
    jobs = []
    acc = {}
    cnt = {}
    for di in doc_list:
        d = DOCS[di]
        acc[di] = np.zeros((d["n"], K), dtype=np.float64)
        cnt[di] = np.zeros(d["n"], dtype=np.float64)
        for L in chunk_lens:
            parts = rr_utils.build_doc_chunks(d["sent_ids"], None, HEADER_IDS[d["domain"]], SEP_ID,
                                              L, DOC_CHUNK_CTX, MAX_SENT_TOK)
            for c in parts:
                c["doc"] = di
                jobs.append(c)
    jobs.sort(key=lambda c: len(c["input_ids"]))

    pos = 0
    while pos < len(jobs):
        batch_chunks = jobs[pos:pos + EVAL_BATCH]
        batch = to_device(collate(batch_chunks))
        with torch.no_grad():
            em = model(batch)
            logp = torch.log_softmax(em, dim=-1).cpu().numpy()
        for b in range(len(batch_chunks)):
            c = batch_chunks[b]
            for k in range(len(c["end_pos"])):
                if c["is_target"][k]:
                    s = int(c["sent_idx"][k])
                    acc[c["doc"]][s] += logp[b, k]
                    cnt[c["doc"]][s] += 1
        pos += EVAL_BATCH

    out = {}
    for di in doc_list:
        if (cnt[di] < 1).any():
            raise RuntimeError("some sentences got no prediction in doc " + DOCS[di]["id"])
        avg = acc[di] / cnt[di][:, None]
        m = avg.max(axis=1, keepdims=True)
        avg = avg - (m + np.log(np.exp(avg - m).sum(axis=1, keepdims=True)))
        out[di] = avg.astype(np.float32)
    return out


def val_macro_f1(model, val_idx):
    preds = predict_docs(model, val_idx, [INFER_CHUNK_LENS[0]])
    conf = np.zeros((K, K), dtype=np.int64)
    for di in val_idx:
        conf += confusion_matrix(DOCS[di]["labels"], preds[di].argmax(axis=1))
    return macro_from_conf(conf)["f1_present"]


def train_fold(k, train_idx, val_idx, fold_dir, teacher=None, extra_idx=None, seed_shift=0, tag="A"):
    os.makedirs(fold_dir, exist_ok=True)
    last_path = os.path.join(fold_dir, "last.pt")
    best_path = os.path.join(fold_dir, "best.pt")
    result_path = os.path.join(fold_dir, "fold_result.json")
    use_kd = teacher is not None
    train_all = list(train_idx)
    if extra_idx is not None:
        train_all = train_all + list(extra_idx)

    counts = class_counts(train_idx)
    weights = make_class_weights(counts)
    rare_labels = []
    for c in range(K):
        if 0 < counts[c] <= RARE_MAX_COUNT:
            rare_labels.append(c)
    print("stage", tag, "| fold", k, "| train docs", len(train_idx), "| val docs", len(val_idx),
          "| unlabelled docs for distillation", 0 if extra_idx is None else len(extra_idx))
    print("  class weights:", {LABEL_NAMES[c]: round(float(weights[c]), 2) for c in range(K) if counts[c] > 0})
    print("  rare labels (oversampled):", [LABEL_NAMES[c] for c in rare_labels],
          "| labels never seen in training:", [LABEL_NAMES[c] for c in range(K) if counts[c] == 0])
    weight_tensor = torch.tensor(weights, dtype=torch.float32, device=DEVICE)

    set_seed(SEED + 100 * k + seed_shift)
    model = build_model(identity_check=False)

    lora_params = []
    head_params = []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "lora_" in name:
            lora_params.append(p)
        else:
            head_params.append(p)
    optimizer = torch.optim.AdamW(
        [{"params": lora_params, "lr": LR_LORA}, {"params": head_params, "lr": LR_HEAD}],
        weight_decay=WEIGHT_DECAY)

    # exponential moving average of the trainable weights (evaluated and saved instead of the raw weights)
    ema = {}
    if EMA_DECAY > 0:
        for name, p in model.named_parameters():
            if p.requires_grad:
                ema[name] = p.detach().clone()
    ema_updates = 0

    rng0 = random.Random(SEED * 1000 + 100 * k + seed_shift)
    est_chunks = len(make_train_chunks(train_all, rng0, rare_labels, teacher))
    steps_per_epoch = max(1, math.ceil(est_chunks / BATCH_CHUNKS))
    total_steps = steps_per_epoch * EPOCHS
    warm = max(1, int(WARMUP_FRAC * total_steps))

    def lr_lambda(step):
        if step < warm:
            return (step + 1) / warm
        progress = min(1.0, (step - warm) / max(1, total_steps - warm))
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    state = {"epoch": 0, "best_f1": -1.0, "best_epoch": -1, "bad": 0, "history": []}
    if os.path.exists(last_path):
        ck = torch.load(last_path, map_location="cpu", weights_only=False)
        load_trainable_state(model, ck["model"])
        optimizer.load_state_dict(ck["opt"])
        scheduler.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"])
        state = ck["state"]
        if EMA_DECAY > 0 and "ema" in ck:
            for name in ema:
                ema[name].copy_(ck["ema"][name].to(DEVICE))
            ema_updates = int(ck.get("ema_updates", 0))
        print("  resumed fold", k, "at epoch", state["epoch"], "| best F1 so far %.4f" % state["best_f1"])

    def swap_in_ema():
        backup = {}
        for name, p in model.named_parameters():
            if name in ema:
                backup[name] = p.data.clone()
                p.data.copy_(ema[name])
        return backup

    def swap_back(backup):
        for name, p in model.named_parameters():
            if name in backup:
                p.data.copy_(backup[name])

    paused = False
    while state["epoch"] < EPOCHS:
        epoch = state["epoch"]
        if time_is_up():
            paused = True
            break
        t_epoch = time.time()
        set_seed(SEED + 100 * k + epoch + seed_shift)
        rng = random.Random(SEED * 1000 + 100 * k + epoch + 1 + seed_shift)
        chunks = make_train_chunks(train_all, rng, rare_labels, teacher)
        model.train()
        loss_total = 0.0
        n_steps = 0
        pos = 0
        while pos < len(chunks):
            big = chunks[pos:pos + BATCH_CHUNKS]
            pos += BATCH_CHUNKS

            wsum = 0.0
            ksum = 0.0
            for c in big:
                valid = c["labels"][c["labels"] >= 0]
                if len(valid) > 0:
                    wsum += float(weights[valid].sum())
                if use_kd and "teacher" in c:
                    ksum += float(c["is_target"].sum())
            if wsum <= 0 and ksum <= 0:
                continue

            optimizer.zero_grad(set_to_none=True)
            step_loss = 0.0
            for m_start in range(0, len(big), MICRO_BATCH):
                micro = big[m_start:m_start + MICRO_BATCH]
                batch = to_device(collate(micro))
                em = model(batch)
                hard_w = (1.0 - KD_LAMBDA) if use_kd else 1.0
                loss = None
                if wsum > 0:
                    loss_sum = F.cross_entropy(em.reshape(-1, K), batch["labels"].reshape(-1), weight=weight_tensor,
                                               ignore_index=-100, reduction="sum", label_smoothing=LABEL_SMOOTHING)
                    loss = hard_w * loss_sum / wsum
                if use_kd and ksum > 0 and "teacher" in batch:
                    logq = F.log_softmax(em / KD_TEMP, dim=-1)
                    pt = F.softmax(batch["teacher"] / KD_TEMP, dim=-1)
                    kl = (pt * (torch.log(pt.clamp_min(1e-8)) - logq)).sum(dim=-1)
                    kd_sum = (kl * batch["tmask"].float()).sum() * (KD_TEMP * KD_TEMP)
                    kd_loss = KD_LAMBDA * kd_sum / ksum
                    loss = kd_loss if loss is None else loss + kd_loss
                if loss is None:
                    continue
                scaler.scale(loss).backward()
                step_loss += float(loss.detach())
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(lora_params + head_params, GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            loss_total += step_loss
            n_steps += 1

            if EMA_DECAY > 0:
                ema_updates += 1
                decay = min(EMA_DECAY, (1.0 + ema_updates) / (10.0 + ema_updates))
                with torch.no_grad():
                    for name, p in model.named_parameters():
                        if name in ema:
                            ema[name].mul_(decay).add_(p.detach(), alpha=1.0 - decay)

        backup = swap_in_ema() if EMA_DECAY > 0 else None
        f1 = val_macro_f1(model, val_idx)
        improved = f1 > state["best_f1"] + 1e-6
        if improved:
            state["best_f1"] = f1
            state["best_epoch"] = epoch + 1
            state["bad"] = 0
            torch.save(get_trainable_state(model), best_path)      # EMA weights when EMA is on
        else:
            state["bad"] += 1
        if backup is not None:
            swap_back(backup)
        state["history"].append({"epoch": epoch + 1, "train_loss": loss_total / max(n_steps, 1), "val_macro_f1": f1})
        state["epoch"] = epoch + 1
        print("  fold %d epoch %d/%d | loss %.4f | val macro-F1 %.4f%s | %.0fs" % (
            k, epoch + 1, EPOCHS, loss_total / max(n_steps, 1), f1, "  *best*" if improved else "",
            time.time() - t_epoch))

        ck = {"model": get_trainable_state(model), "opt": optimizer.state_dict(),
              "sched": scheduler.state_dict(), "scaler": scaler.state_dict(), "state": state}
        if EMA_DECAY > 0:
            ck["ema"] = {name: ema[name].detach().cpu().clone() for name in ema}
            ck["ema_updates"] = ema_updates
        torch.save(ck, last_path)
        if state["bad"] >= PATIENCE:
            print("  early stop (no improvement for", PATIENCE, "epochs)")
            break

    del optimizer, scheduler
    if paused:
        print("  session time limit reached: fold", k, "paused at epoch", state["epoch"])
        return None

    result = {"fold": k, "stage": tag, "best_epoch": state["best_epoch"], "best_val_macro_f1": state["best_f1"],
              "history": state["history"], "train_docs": len(train_idx), "val_docs": len(val_idx)}
    with open(result_path, "w") as fh:
        json.dump(result, fh, indent=2)
    if os.path.exists(last_path):
        os.remove(last_path)          # keep the output small, best.pt is what we need
    return model

## 8. Stage A (teacher folds) and stage B (distilled students)

In [ ]:
# ---------- fold assignment ----------
def make_folds():
    labelled = list(range(N_LAB))
    if PROTOCOL == "dev":
        train_idx = []
        val_idx = []
        for i in labelled:
            if DOCS[i]["split"] == "dev":
                val_idx.append(i)
            else:
                train_idx.append(i)
        if len(val_idx) == 0:
            raise RuntimeError("no dev documents")
        return [(train_idx, val_idx)]

    rng = random.Random(SEED)
    fold_of = {}
    for domain in DOMAINS:
        idx = []
        for i in labelled:
            if DOCS[i]["domain"] == domain:
                idx.append(i)
        idx.sort(key=lambda i: -DOCS[i]["n"])          # big documents are spread evenly over the folds
        for block_start in range(0, len(idx), N_FOLDS):
            block = idx[block_start:block_start + N_FOLDS]
            order = list(range(N_FOLDS))
            rng.shuffle(order)
            for j in range(len(block)):
                fold_of[block[j]] = order[j]
    folds = []
    for k in range(N_FOLDS):
        train_idx = []
        val_idx = []
        for i in labelled:
            if fold_of[i] == k:
                val_idx.append(i)
            else:
                train_idx.append(i)
        if len(val_idx) == 0:
            raise RuntimeError("empty validation fold, lower N_FOLDS")
        folds.append((train_idx, val_idx))
    return folds


fingerprint_text = PROTOCOL + str(N_FOLDS) + str(SEED) + str(SMOKE_TEST)
for i in range(N_LAB):
    fingerprint_text += DOCS[i]["domain"] + DOCS[i]["id"] + str(DOCS[i]["n"])
FINGERPRINT = hashlib.md5(fingerprint_text.encode("utf-8")).hexdigest()

state_path = os.path.join(OUT_DIR, "cv_state.json")
if os.path.exists(state_path):
    old = json.load(open(state_path))
    if old.get("fingerprint") != FINGERPRINT:
        raise RuntimeError(
            "The labelled documents / protocol differ from the earlier run in OUT_DIR (the dataset may have changed). "
            "Remove the old fold folders or change EXPERIMENT_NAME to start a new run.")

FOLDS = make_folds()
with open(state_path, "w") as fh:
    json.dump({"fingerprint": FINGERPRINT, "protocol": PROTOCOL, "n_folds": len(FOLDS), "seed": SEED,
               "folds": [[DOCS[i]["id"] for i in v] for (t, v) in FOLDS]}, fh)
for k in range(len(FOLDS)):
    dom_counts = {}
    for i in FOLDS[k][1]:
        dom_counts[DOCS[i]["domain"]] = dom_counts.get(DOCS[i]["domain"], 0) + 1
    print("fold", k, "| train docs", len(FOLDS[k][0]), "| val docs", len(FOLDS[k][1]), dom_counts)


def save_pickle(obj, path):
    fh = open(path, "wb")
    pickle.dump(obj, fh, protocol=4)
    fh.close()


def load_pickle(path):
    fh = open(path, "rb")
    obj = pickle.load(fh)
    fh.close()
    return obj



# ---------- stage A may come from an earlier run with the same folds (it is then the teacher, nothing is retrained) ----------
def run_has_stage(root, prefix):
    for k in range(len(FOLDS)):
        for name in ["best.pt", "fold_result.json", "oof_logp.pkl"]:
            if not os.path.exists(os.path.join(root, prefix + str(k), name)):
                return False
    return True


def run_has_test(root, prefix):
    for k in range(len(FOLDS)):
        if not os.path.exists(os.path.join(root, prefix + str(k), "test_logp.pkl")):
            return False
    return True


own_started = os.path.exists(os.path.join(OUT_DIR, "fold0", "best.pt"))
if USE_TEACHER_RUN and not own_started:
    found_runs = []
    state_files = find_files("cv_state.json")
    if TEACHER_DIR is not None:
        state_files = [os.path.join(TEACHER_DIR, "cv_state.json")]
    for p in state_files:
        root = os.path.dirname(p)
        if os.path.abspath(root) == os.path.abspath(OUT_DIR):
            continue
        try:
            st = json.load(open(p))
        except Exception:
            continue
        if st.get("fingerprint") != FINGERPRINT:
            continue
        if not run_has_stage(root, "fold"):
            continue
        found_runs.append((1 if run_has_test(root, "fold") else 0, os.path.getmtime(p), root))
    if len(found_runs) > 0:
        found_runs.sort()
        teacher_root = found_runs[-1][2]
        for k in range(len(FOLDS)):
            src_dir = os.path.join(teacher_root, "fold" + str(k))
            dst_dir = os.path.join(OUT_DIR, "fold" + str(k))
            os.makedirs(dst_dir, exist_ok=True)
            for name in ["best.pt", "fold_result.json", "oof_logp.pkl", "test_logp.pkl"]:
                if os.path.exists(os.path.join(src_dir, name)):
                    shutil.copy(os.path.join(src_dir, name), os.path.join(dst_dir, name))
        print("STAGE A taken from an earlier run with identical folds (it is the teacher, no retraining):", teacher_root)
        print("   (its test predictions are %s)" % ("present" if found_runs[-1][0] else "missing: they will be computed from its best.pt"))
    else:
        print("No earlier run with identical folds found: stage A is trained here.")


def run_stage(prefix, tag, teacher=None, extra_idx=None, seed_shift=0):
    stage_done = True
    test_ids = [DOCS[i]["id"] for i in TEST_IDX]
    test_n = [DOCS[i]["n"] for i in TEST_IDX]
    for k in range(len(FOLDS)):
        train_idx, val_idx = FOLDS[k]
        fold_dir = os.path.join(OUT_DIR, prefix + str(k))
        result_path = os.path.join(fold_dir, "fold_result.json")
        oof_path = os.path.join(fold_dir, "oof_logp.pkl")
        test_path = os.path.join(fold_dir, "test_logp.pkl")

        model = None
        if not os.path.exists(result_path):
            model = train_fold(k, train_idx, val_idx, fold_dir, teacher, extra_idx, seed_shift, tag)
            if model is None:
                stage_done = False
                break
        else:
            print("stage", tag, "fold", k, "already trained, skipping training")

        need_oof = not os.path.exists(oof_path)
        need_test = False
        if N_TEST > 0:
            if not os.path.exists(test_path):
                need_test = True
            else:
                old_pack = load_pickle(test_path)
                if old_pack["ids"] != test_ids or ("n" in old_pack and old_pack["n"] != test_n):
                    print("  fold", k, "old test predictions belong to a different test set, recomputing")
                    need_test = True
        if need_oof or need_test:
            if model is None:
                set_seed(SEED + 100 * k + seed_shift)
                model = build_model(identity_check=False)
            best_state = torch.load(os.path.join(fold_dir, "best.pt"), map_location="cpu")
            load_trainable_state(model, best_state)
            if need_oof:
                t1 = time.time()
                oof = predict_docs(model, val_idx, INFER_CHUNK_LENS)
                save_pickle({"ids": [DOCS[i]["id"] for i in val_idx], "logp": oof}, oof_path)
                print("  stage", tag, "fold", k, "OOF predictions saved (%.0fs)" % (time.time() - t1))
            if need_test:
                t1 = time.time()
                tst = predict_docs(model, TEST_IDX, INFER_CHUNK_LENS)
                save_pickle({"ids": test_ids, "n": test_n, "logp": tst}, test_path)
                print("  stage", tag, "fold", k, "test predictions saved (%.0fs)" % (time.time() - t1))
        if model is not None:
            del model
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
    return stage_done


def norm_logp(x):
    m = x.max(axis=1, keepdims=True)
    return x - (m + np.log(np.exp(x - m).sum(axis=1, keepdims=True)))


def build_teacher():
    # soft labels: out-of-fold log-probs for labelled documents, fold-averaged log-probs for the test documents
    teacher = {}
    for k in range(len(FOLDS)):
        pack = load_pickle(os.path.join(OUT_DIR, "fold" + str(k), "oof_logp.pkl"))
        for di in FOLDS[k][1]:
            teacher[di] = pack["logp"][di].astype(np.float32)
    if KD_USE_TEST and N_TEST > 0:
        packs = []
        for k in range(len(FOLDS)):
            packs.append(load_pickle(os.path.join(OUT_DIR, "fold" + str(k), "test_logp.pkl")))
        for di in TEST_IDX:
            total = np.zeros((DOCS[di]["n"], K), dtype=np.float64)
            for pack in packs:
                total += pack["logp"][di]
            teacher[di] = norm_logp(total / len(packs)).astype(np.float32)
    return teacher


ALL_DONE = run_stage("fold", "A")
if ALL_DONE and USE_KD:
    print("\n=== stage B: distilled students (teacher = stage A out-of-fold / fold-averaged log-probs) ===")
    TEACHER = build_teacher()
    EXTRA = TEST_IDX if (KD_USE_TEST and N_TEST > 0) else None
    ALL_DONE = run_stage("kd_fold", "B", TEACHER, EXTRA, seed_shift=7)

if ALL_DONE:
    print("\nAll folds finished.")
else:
    print("\nSTOPPED because of MAX_SESSION_HOURS. Progress is saved in", OUT_DIR)
    print("Attach this notebook's output as an input to a new run of the same notebook to continue.")

## 9. Post-processing, second stage and held-out comparison

Candidates: source (**A** teacher / **B** distilled student / **AB** average) x stage (**S1** the log-probs as they are / **S2** document-level BiGRU) x class bias (off / on). `tau` (prior correction) and `alpha` (transition weight in Viterbi) are always tuned; the class bias adds one value per label and domain. For every candidate the tuning is done on 4 folds and scored on the 5th (leave-one-fold-out), so the table is an honest comparison and the winner is chosen from it.

In [ ]:
if not ALL_DONE:
    print("Skipped: not all folds are finished yet.")
else:
    t_post = time.time()

    # ---------- the candidate sources of log-probs ----------
    def load_source(prefix):
        oof_entries = []
        test_by_fold = []
        for k in range(len(FOLDS)):
            pack = load_pickle(os.path.join(OUT_DIR, prefix + str(k), "oof_logp.pkl"))
            for di in FOLDS[k][1]:
                oof_entries.append({"doc": di, "fold": k, "logp": pack["logp"][di].astype(np.float64)})
            if N_TEST > 0:
                tp = load_pickle(os.path.join(OUT_DIR, prefix + str(k), "test_logp.pkl"))
                test_by_fold.append({di: tp["logp"][di].astype(np.float64) for di in TEST_IDX})
        return {"oof": oof_entries, "test": test_by_fold}

    SOURCES = {"A": load_source("fold")}
    SOURCE_TEXT = {"A": "stage A (teacher)"}
    if USE_KD:
        SOURCES["B"] = load_source("kd_fold")
        SOURCE_TEXT["B"] = "stage B (distilled student)"
        b_by_doc = {}
        for e in SOURCES["B"]["oof"]:
            b_by_doc[e["doc"]] = e["logp"]
        blend_oof = []
        for e in SOURCES["A"]["oof"]:
            blend_oof.append({"doc": e["doc"], "fold": e["fold"], "logp": norm_logp(0.5 * (e["logp"] + b_by_doc[e["doc"]]))})
        blend_test = []
        for k in range(len(SOURCES["A"]["test"])):
            one = {}
            for di in TEST_IDX:
                one[di] = norm_logp(0.5 * (SOURCES["A"]["test"][k][di] + SOURCES["B"]["test"][k][di]))
            blend_test.append(one)
        SOURCES["AB"] = {"oof": blend_oof, "test": blend_test}
        SOURCE_TEXT["AB"] = "A + B averaged"

    # ---------- transitions, priors ----------
    fold_trans = []
    for k in range(len(FOLDS)):
        tr = {}
        for domain in DOMAINS:
            tr[domain] = log_transitions(FOLDS[k][0], domain)
        fold_trans.append(tr)
    all_lab = list(range(N_LAB))
    final_trans = {}
    DOM_PRIOR = {}
    DOM_SEEN = {}
    for domain in DOMAINS:
        final_trans[domain] = log_transitions(all_lab, domain)
        idx = [i for i in all_lab if DOCS[i]["domain"] == domain]
        cnt = class_counts(idx)
        DOM_PRIOR[domain] = log_prior_from_counts(cnt)
        DOM_SEEN[domain] = cnt > 0

    def bias_from_tau(tau, domain):
        return np.clip(-tau * DOM_PRIOR[domain], -BIAS_CLIP, BIAS_CLIP)

    def decode_b(logp, trans_log, bias, alpha):
        emission = logp + bias[None, :]
        n = emission.shape[0]
        if alpha == 0.0 or n == 1:
            return emission.argmax(axis=1)
        score = emission[0].copy()
        back = np.zeros((n, K), dtype=np.int64)
        pair = alpha * trans_log
        for i in range(1, n):
            cand = score[:, None] + pair
            back[i] = cand.argmax(axis=0)
            score = cand.max(axis=0) + emission[i]
        path = np.zeros(n, dtype=np.int64)
        path[n - 1] = int(score.argmax())
        for i in range(n - 1, 0, -1):
            path[i - 1] = back[i, path[i]]
        return path

    def eval_entries(entries, theta, domain=None):
        conf = np.zeros((K, K), dtype=np.int64)
        for e in entries:
            d = DOCS[e["doc"]]
            if domain is not None and d["domain"] != domain:
                continue
            pred = decode_b(e["logp"], fold_trans[e["fold"]][d["domain"]], theta["bias"][d["domain"]], theta["alpha"])
            conf += confusion_matrix(d["labels"], pred)
        return conf

    def copy_theta(theta):
        return {"alpha": theta["alpha"], "bias": {dom: theta["bias"][dom].copy() for dom in theta["bias"]}}

    def fit_theta(entries, use_bias):
        # step 1: one tau (prior correction) and one alpha (transition weight) for everything
        best = None
        for tau in TAU_GRID:
            for alpha in ALPHA_GRID:
                theta = {"alpha": alpha, "bias": {dom: bias_from_tau(tau, dom) for dom in DOMAINS}}
                sc = macro_from_conf(eval_entries(entries, theta))["f1_present"]
                better = False
                if best is None or sc > best[0] + 1e-6:
                    better = True
                elif abs(sc - best[0]) <= 1e-6 and (abs(tau) + alpha) < best[3]:
                    better = True
                if better:
                    best = (sc, theta, tau, abs(tau) + alpha)
        theta = best[1]
        theta["tau"] = best[2]
        if not use_bias:
            return theta

        # step 2: per-class, per-domain bias, coordinate ascent on the pooled macro-F1
        conf_dom = {}
        for dom in DOMAINS:
            conf_dom[dom] = eval_entries(entries, theta, dom)
        cur = macro_from_conf(sum(conf_dom.values()))["f1_present"]
        for _round in range(BIAS_ROUNDS):
            changed = False
            for dom in DOMAINS:
                for c in range(K):
                    if not DOM_SEEN[dom][c]:
                        continue
                    best_gain = 0.0
                    best_val = None
                    best_conf = None
                    for delta in BIAS_STEPS:
                        new_val = float(np.clip(theta["bias"][dom][c] + delta, -BIAS_CLIP, BIAS_CLIP))
                        if new_val == theta["bias"][dom][c]:
                            continue
                        old_val = theta["bias"][dom][c]
                        theta["bias"][dom][c] = new_val
                        conf_new = eval_entries(entries, theta, dom)
                        theta["bias"][dom][c] = old_val
                        total = conf_new.copy()
                        for other in DOMAINS:
                            if other != dom:
                                total = total + conf_dom[other]
                        gain = macro_from_conf(total)["f1_present"] - cur
                        if gain > best_gain:
                            best_gain = gain
                            best_val = new_val
                            best_conf = conf_new
                    if best_val is not None and best_gain >= BIAS_MIN_GAIN:
                        theta["bias"][dom][c] = best_val
                        conf_dom[dom] = best_conf
                        cur += best_gain
                        changed = True
            if not changed:
                break
        return theta

    def honest_score(entries, use_bias):
        # leave-one-fold-out: tune on the other folds, score on the held-out fold
        folds_here = sorted(set(e["fold"] for e in entries))
        if len(folds_here) < 2:
            theta = fit_theta(entries, use_bias)
            conf = eval_entries(entries, theta)
            return macro_from_conf(conf), conf, False
        conf = np.zeros((K, K), dtype=np.int64)
        for f in folds_here:
            theta = fit_theta([e for e in entries if e["fold"] != f], use_bias)
            conf += eval_entries([e for e in entries if e["fold"] == f], theta)
        return macro_from_conf(conf), conf, True

    # ---------- stage 2: document-level BiGRU on the log-probs ----------
    class S2Net(nn.Module):
        def __init__(self):
            super().__init__()
            in_dim = K + 1 + 4 + len(DOMAINS)
            self.inp = nn.Linear(in_dim, S2_HIDDEN)
            self.drop = nn.Dropout(S2_DROPOUT)
            self.gru = nn.GRU(S2_HIDDEN, S2_HIDDEN, batch_first=True, bidirectional=True)
            self.out = nn.Linear(3 * S2_HIDDEN, K)
            nn.init.zeros_(self.out.weight)
            nn.init.zeros_(self.out.bias)

        def forward(self, x, aux, lengths):
            S = x.shape[1]
            if self.training and S2_NOISE > 0:
                x = torch.log_softmax(x + torch.randn_like(x) * S2_NOISE, dim=-1)
            ent = -(x.exp() * x).sum(dim=-1, keepdim=True)
            feat = torch.cat([x.clamp(min=-12.0) / 4.0, ent, aux], dim=-1)
            h = self.drop(F.gelu(self.inp(feat)))
            packed = nn.utils.rnn.pack_padded_sequence(h, lengths.cpu(), batch_first=True, enforce_sorted=False)
            o, _ = self.gru(packed)
            o, _ = nn.utils.rnn.pad_packed_sequence(o, batch_first=True, total_length=S)
            delta = self.out(self.drop(torch.cat([o, h], dim=-1)))
            return torch.log_softmax(x + delta, dim=-1)       # starts exactly at the input log-probs

    def s2_batch(items):
        B = len(items)
        S = max(len(lp) for _, lp in items)
        x = np.full((B, S, K), math.log(1.0 / K), dtype=np.float32)
        aux = np.zeros((B, S, 4 + len(DOMAINS)), dtype=np.float32)
        y = np.full((B, S), -100, dtype=np.int64)
        lengths = np.zeros(B, dtype=np.int64)
        for b in range(B):
            di, lp = items[b]
            n = len(lp)
            x[b, :n] = lp
            lengths[b] = n
            aux[b, :n, :4] = DOCS[di]["feats"][:, N_CUE:N_CUE + 4]
            aux[b, :n, 4 + DOMAIN_IDX[DOCS[di]["domain"]]] = 1.0
            if DOCS[di]["labels"] is not None:
                y[b, :n] = DOCS[di]["labels"]
        return (torch.from_numpy(x).to(DEVICE), torch.from_numpy(aux).to(DEVICE),
                torch.from_numpy(y).to(DEVICE), torch.from_numpy(lengths))

    def s2_train(train_entries, seed):
        torch.manual_seed(seed)
        rng = random.Random(seed)
        net = S2Net().to(DEVICE)
        counts = class_counts([e["doc"] for e in train_entries])
        w = torch.tensor(make_class_weights(counts), dtype=torch.float32, device=DEVICE)
        opt = torch.optim.AdamW(net.parameters(), lr=S2_LR, weight_decay=1e-2)
        n_batches = max(1, math.ceil(len(train_entries) / S2_BATCH))
        total = n_batches * S2_EPOCHS
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 0.5 * (1.0 + math.cos(math.pi * min(1.0, s / max(1, total)))))
        items = [(e["doc"], e["logp"].astype(np.float32)) for e in train_entries]
        for _ep in range(S2_EPOCHS):
            net.train()
            rng.shuffle(items)
            for i in range(0, len(items), S2_BATCH):
                x, aux, y, lengths = s2_batch(items[i:i + S2_BATCH])
                out = net(x, aux, lengths)
                loss = F.cross_entropy(out.reshape(-1, K), y.reshape(-1), weight=w, ignore_index=-100, label_smoothing=0.02)
                opt.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
                opt.step()
                sched.step()
        net.eval()
        return net

    def s2_apply(nets, di, logp):
        x, aux, y, lengths = s2_batch([(di, logp.astype(np.float32))])
        probs = None
        with torch.no_grad():
            for net in nets:
                p = net(x, aux, lengths).exp().cpu().numpy()[0, :len(logp)].astype(np.float64)
                probs = p if probs is None else probs + p
        return norm_logp(np.log(np.clip(probs / len(nets), 1e-12, None)))

    def s2_nested(entries):
        # out-of-fold outputs of the second stage: train on the other folds, predict the held-out fold
        folds_here = sorted(set(e["fold"] for e in entries))
        out = []
        for f in folds_here:
            train_e = [e for e in entries if e["fold"] != f]
            nets = [s2_train(train_e, SEED + 31 * f + s) for s in range(S2_SEEDS)]
            for e in entries:
                if e["fold"] == f:
                    out.append({"doc": e["doc"], "fold": f, "logp": s2_apply(nets, e["doc"], e["logp"])})
            del nets
        return out

    # ---------- evaluate every configuration honestly ----------
    results = []
    S2_OOF = {}
    for src in SOURCES:
        entries_s1 = SOURCES[src]["oof"]
        if USE_STAGE2:
            if len(set(e["fold"] for e in entries_s1)) >= 2:
                S2_OOF[src] = s2_nested(entries_s1)
            else:
                print("stage 2 needs at least 2 folds, skipped for", src)
        for stage in ["S1", "S2"]:
            if stage == "S2" and src not in S2_OOF:
                continue
            entries = entries_s1 if stage == "S1" else S2_OOF[src]
            for use_bias in ([False, True] if USE_CLASS_BIAS else [False]):
                t1 = time.time()
                sc, conf, is_honest = honest_score(entries, use_bias)
                complexity = (2 if stage == "S2" else 0) + (1 if use_bias else 0) + (1 if src != "A" else 0)
                results.append({"source": src, "stage": stage, "bias": use_bias, "f1": sc["f1_present"],
                                "precision": sc["precision_present"], "recall": sc["recall_present"],
                                "complexity": complexity, "conf": conf, "honest": is_honest})
                print("  %-3s %-2s bias=%-5s -> held-out macro-F1 %.4f | P %.4f | R %.4f | %.0fs" % (
                    src, stage, str(use_bias), sc["f1_present"], sc["precision_present"], sc["recall_present"], time.time() - t1))

    results.sort(key=lambda r: -(r["f1"] - 0.001 * r["complexity"]))
    BEST = results[0]
    if not all(r["honest"] for r in results):
        print("\nNOTE: only one fold (PROTOCOL = dev), so these scores are tuned and scored on the same documents (optimistic).")
    print("\nHeld-out macro-F1 of every configuration (tuning is always done on the other folds):")
    print("%-26s %-3s %-2s %-6s %8s %8s %8s" % ("source", "src", "st", "bias", "F1", "P", "R"))
    for r in results:
        print("%-26s %-3s %-2s %-6s %8.4f %8.4f %8.4f" % (SOURCE_TEXT[r["source"]], r["source"], r["stage"], str(r["bias"]),
                                                      r["f1"], r["precision"], r["recall"]))
    print("\nCHOSEN: source %s, %s, class bias %s -> held-out macro-F1 %.4f (post-processing took %.0fs)" % (
        BEST["source"], BEST["stage"], BEST["bias"], BEST["f1"], time.time() - t_post))
    base_row = None
    for r in results:
        if r["source"] == "A" and r["stage"] == "S1" and not r["bias"]:
            base_row = r
    if base_row is not None:
        print("  (stage A with only tau/alpha tuning, the old recipe: %.4f)" % base_row["f1"])

    # ---------- final parameters for the chosen configuration ----------
    best_entries = SOURCES[BEST["source"]]["oof"] if BEST["stage"] == "S1" else S2_OOF[BEST["source"]]
    THETA = fit_theta(best_entries, BEST["bias"])
    print("\nFinal alpha = %.2f | tau (start of the bias search) = %.2f" % (THETA["alpha"], THETA.get("tau", 0.0)))
    for dom in DOMAINS:
        moved = {LABEL_NAMES[c]: round(float(THETA["bias"][dom][c]), 2) for c in range(K) if abs(THETA["bias"][dom][c]) > 1e-9}
        print("  bias", dom, moved)

    final_conf = BEST["conf"]
    final = macro_from_conf(final_conf)
    precision, recall, f1, support = scores_from_conf(final_conf)
    print("\nPooled held-out result of the chosen configuration: macro-F1 present %.4f | all-13 %.4f | macro-P %.4f | macro-R %.4f" % (
        final["f1_present"], final["f1_all13"], final["precision_present"], final["recall_present"]))
    print("\n%-24s %8s %8s %8s %8s" % ("label", "support", "prec", "recall", "F1"))
    for c in range(K):
        print("%-24s %8d %8.3f %8.3f %8.3f" % (LABEL_NAMES[c], support[c], precision[c], recall[c], f1[c]))

    pairs = []
    for a in range(K):
        for b in range(K):
            if a != b and final_conf[a, b] > 0:
                pairs.append((int(final_conf[a, b]), LABEL_NAMES[a], LABEL_NAMES[b]))
    pairs.sort(reverse=True)
    print("\nMost frequent confusions (true -> predicted):")
    for n, a, b in pairs[:10]:
        print("   %5d  %s -> %s" % (n, a, b))

    fold_summaries = []
    for k in range(len(FOLDS)):
        entry = {"A": json.load(open(os.path.join(OUT_DIR, "fold" + str(k), "fold_result.json")))}
        if USE_KD:
            entry["B"] = json.load(open(os.path.join(OUT_DIR, "kd_fold" + str(k), "fold_result.json")))
        fold_summaries.append(entry)
    report = {
        "protocol": PROTOCOL, "n_folds": len(FOLDS),
        "chosen": {"source": BEST["source"], "stage": BEST["stage"], "class_bias": BEST["bias"], "heldout_macro_f1": BEST["f1"],
                   "alpha": THETA["alpha"], "tau_start": THETA.get("tau", 0.0),
                   "bias": {dom: {LABEL_NAMES[c]: float(THETA["bias"][dom][c]) for c in range(K)} for dom in DOMAINS}},
        "all_configs": [{"source": r["source"], "stage": r["stage"], "class_bias": r["bias"], "macro_f1": r["f1"],
                         "macro_p": r["precision"], "macro_r": r["recall"], "heldout": r["honest"]} for r in results],
        "pooled": final,
        "per_class": {LABEL_NAMES[c]: {"support": int(support[c]), "precision": float(precision[c]),
                                       "recall": float(recall[c]), "f1": float(f1[c])} for c in range(K)},
        "top_confusions": [{"count": n, "true": a, "pred": b} for n, a, b in pairs[:25]],
        "confusion_matrix": final_conf.tolist(), "labels": LABEL_NAMES,
        "folds": fold_summaries,
        "settings": {"EPOCHS": EPOCHS, "LR_LORA": LR_LORA, "EMA_DECAY": EMA_DECAY, "LORA_DROPOUT": LORA_DROPOUT,
                     "HEAD_DROPOUT": HEAD_DROPOUT, "USE_KD": USE_KD, "KD_LAMBDA": KD_LAMBDA, "KD_TEMP": KD_TEMP,
                     "KD_USE_TEST": KD_USE_TEST, "USE_CLASS_BIAS": USE_CLASS_BIAS, "USE_STAGE2": USE_STAGE2,
                     "HEAD": HEAD, "USE_FEATURES": USE_FEATURES, "CLASS_WEIGHTS": CLASS_WEIGHTS,
                     "INFER_CHUNK_LENS": INFER_CHUNK_LENS, "SMOKE_TEST": SMOKE_TEST},
    }
    with open(os.path.join(OUT_DIR, "oof_report.json"), "w") as fh:
        json.dump(report, fh, indent=2)
    print("\nSaved oof_report.json")

## 10. Test predictions

In [ ]:
if not ALL_DONE:
    print("Skipped: not all folds are finished yet.")
elif N_TEST == 0:
    print("No test documents in the data file. Re-run the data-prep notebook after the test set is released,")
    print("attach its new output and this notebook's output to a new run, and run it again (only test predictions are produced).")
else:
    src_name = BEST["source"]
    final_nets = None
    if BEST["stage"] == "S2":
        final_nets = [s2_train(SOURCES[src_name]["oof"], SEED + 997 + s) for s in range(S2_SEEDS)]

    predictions = {}
    by_domain = {}
    n_fold_models = len(SOURCES[src_name]["test"])
    for di in TEST_IDX:
        d = DOCS[di]
        total = np.zeros((d["n"], K), dtype=np.float64)
        for k in range(n_fold_models):
            lp = SOURCES[src_name]["test"][k][di]
            if final_nets is not None:
                lp = s2_apply(final_nets, di, lp)
            total += lp
        avg = norm_logp(total / n_fold_models)
        path = decode_b(avg, final_trans[d["domain"]], THETA["bias"][d["domain"]], THETA["alpha"])
        labels = []
        for c in path:
            labels.append(LABEL_NAMES[int(c)])
        if len(labels) != d["n"]:
            raise RuntimeError("length mismatch in " + d["id"])
        for lab in labels:
            if lab not in label2id:
                raise RuntimeError("label not in vocab: " + lab)
        predictions[d["id"]] = labels
        if d["domain"] not in by_domain:
            by_domain[d["domain"]] = {}
        by_domain[d["domain"]][d["id"]] = labels

    if len(predictions) != N_TEST:
        print("WARNING: document ids are not unique across CL and IT, the merged file has", len(predictions), "of", N_TEST)

    for domain in by_domain:
        with open(os.path.join(OUT_DIR, "prediction_" + domain + ".json"), "w") as fh:
            json.dump(by_domain[domain], fh)
    with open(os.path.join(OUT_DIR, "prediction.json"), "w") as fh:
        json.dump(predictions, fh)
    with zipfile.ZipFile(os.path.join(OUT_DIR, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(os.path.join(OUT_DIR, "prediction.json"), "prediction.json")

    counts = np.zeros(K, dtype=np.int64)
    for doc_id in predictions:
        for lab in predictions[doc_id]:
            counts[label2id[lab]] += 1
    print("test documents:", N_TEST, "| sentences:", int(counts.sum()), "| from source", src_name, BEST["stage"],
          "| class bias", BEST["bias"])
    print("predicted label distribution:", {LABEL_NAMES[c]: int(counts[c]) for c in range(K) if counts[c] > 0})
    print("Saved prediction.json, prediction_<domain>.json and submission.zip in", OUT_DIR)

## Notes

- The router of every MoE layer is part of the frozen Phase-1 backbone, so the expert assignment of a token does not change during fine-tuning; no load-balancing loss is used.
- Training and OOF folds use the same seed and fold assignment as the Model 1 notebooks, so `oof_logp.pkl` / `test_logp.pkl` of all runs can be ensembled later.
- Time per epoch is higher than Model 1 (253M parameters, 6 routed experts + 1 shared). If the GPU runs out of memory lower `MICRO_BATCH` to 1; `USE_CHECKPOINT = True` is already on.
- `oof_report.json` has the held-out table of all post-processing candidates, the chosen one with its biases, per-class precision/recall/F1 and the most frequent confusions.
- Ablation rows: `USE_STAGE2=False`, `USE_CLASS_BIAS=False`, `HEAD="linear"`, `USE_FEATURES=False`, `CLASS_WEIGHTS="none"`, `OVERSAMPLE_RARE=False`.